# Analisis de datos Apple Watch

### En este notebook se presentarán las principales funciones desarrolladas en Python para el análisis de los datos extraídos del dispositivo Apple Watch. Estas funciones constituyen la base del proceso de tratamiento, transformación y análisis de la información recopilada por el dispositivo.

##### Posteriormente, se crearán los archivos *.py y el archivo main.py, desde el cual se realizará la llamada a las funciones implementadas.

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

def parseo_data (ruta_archivo):
    """
    Lee un archivo XML exportado por Apple Health y convierte los registros
    de salud en un DataFrame de Pandas.

    La función analiza la estructura del archivo XML, extrae todos los
    elementos de tipo 'Record' junto con sus atributos y genera un DataFrame
    que servirá como base para el posterior proceso de limpieza, transformación
    y análisis de los datos. Se visualizan los primeros datos.

    Parámetros
    ----------
    ruta_archivo : str
        Ruta del archivo XML que contiene los datos exportados desde Apple Health.

    Return
    -------
    pandas.DataFrame
        DataFrame con los registros de salud extraídos del archivo XML.
    """
    # Accedemos al nodo raiz del documento XLM, para acceder a todos los elementos que tiene
    tree = ET.parse(ruta_archivo)
    root = tree.getroot()

    # Creamos lista vacia que almacena los atributos de cada registro de salud encontrado
    records = []
    workouts = []
    workout_stats = []
    workout_events = []
    activity_summaries = []
    correlations = []
    correlation_records = []

    # Records de nivel superior 
    for record in root.findall('Record'):
        registro = record.attrib.copy()
        registro['tipo_elemento'] = 'Record'
        records.append(registro)

    # Workouts + sus hijos 
    for i, workout in enumerate(root.findall('Workout')):
        workout_id = f"workout_{i}"
        w = workout.attrib.copy()
        w['workout_id'] = workout_id
        workouts.append(w)

        for stat in workout.findall('WorkoutStatistics'):
            s = stat.attrib.copy()
            s['workout_id'] = workout_id
            workout_stats.append(s)

        for event in workout.findall('WorkoutEvent'):
            e = event.attrib.copy()
            e['workout_id'] = workout_id
            workout_events.append(e)

    # ActivitySummary
    for summary in root.findall('ActivitySummary'):
        activity_summaries.append(summary.attrib.copy())

    # Correlation + sus Records hijos 
    for i, corr in enumerate(root.findall('Correlation')):
        corr_id = f"correlation_{i}"
        c = corr.attrib.copy()
        c['correlation_id'] = corr_id
        correlations.append(c)

        for rec in corr.findall('Record'):
            r = rec.attrib.copy()
            r['correlation_id'] = corr_id
            correlation_records.append(r)
    
    # Convertimos todo a DataFrames
    df_records = pd.DataFrame(records)
    df_workouts = pd.DataFrame(workouts)
    df_workout_stats = pd.DataFrame(workout_stats)
    df_workout_events = pd.DataFrame(workout_events)
    df_activity_summaries = pd.DataFrame(activity_summaries)
    df_correlations = pd.DataFrame(correlations)
    df_correlation_records = pd.DataFrame(correlation_records)

    return {
        'records': df_records,
        'workouts': df_workouts,
        'workout_stats': df_workout_stats,
        'workout_events': df_workout_events, # No es interesante
        'activity_summaries': df_activity_summaries, # Sumatorio diario de energía quemada
        'correlations': df_correlations, #Vacio
        'correlation_records': df_correlation_records #Vacio
    }



In [12]:
data = parseo_data(r"C:\Users\Usuario\Desktop\AppleWatch-DataInsightsML\data\exportación.xml")

In [32]:
tipos_unicos = data['records']['type'].unique()
print(tipos_unicos)

['HKQuantityTypeIdentifierDietaryWater'
 'HKQuantityTypeIdentifierBodyMassIndex' 'HKQuantityTypeIdentifierHeight'
 'HKQuantityTypeIdentifierBodyMass' 'HKQuantityTypeIdentifierHeartRate'
 'HKQuantityTypeIdentifierOxygenSaturation'
 'HKQuantityTypeIdentifierRespiratoryRate'
 'HKQuantityTypeIdentifierBodyFatPercentage'
 'HKQuantityTypeIdentifierLeanBodyMass'
 'HKQuantityTypeIdentifierStepCount'
 'HKQuantityTypeIdentifierDistanceWalkingRunning'
 'HKQuantityTypeIdentifierBasalEnergyBurned'
 'HKQuantityTypeIdentifierActiveEnergyBurned'
 'HKQuantityTypeIdentifierFlightsClimbed'
 'HKQuantityTypeIdentifierNumberOfTimesFallen'
 'HKQuantityTypeIdentifierAppleExerciseTime'
 'HKQuantityTypeIdentifierDistanceSwimming'
 'HKQuantityTypeIdentifierSwimmingStrokeCount'
 'HKQuantityTypeIdentifierRestingHeartRate'
 'HKQuantityTypeIdentifierVO2Max'
 'HKQuantityTypeIdentifierWalkingHeartRateAverage'
 'HKQuantityTypeIdentifierEnvironmentalAudioExposure'
 'HKQuantityTypeIdentifierHeadphoneAudioExposure'
 'HKQu